# Notebook 4 — Dataset Assembly
### Multimodal Driving Risk Prediction | MSc Business Analytics | Dublin Business School

---

## What This Notebook Does
Integrates all feature groups from Notebooks 1, 2, and 3 into one unified multimodal dataset.
Three additional composite features are engineered here.
A mandatory quality check confirms no target leakage before the dataset is saved.

## Key Steps
1. **Load** prepared metadata (NB1), ego-motion features (NB2), visual features (NB3)
2. **Merge** all tables on `clip_id` using inner joins
3. **Engineer** three composite features
4. **Encode** `lighting_condition` from brightness bins
5. **Check** that `anomaly_score` is absent — mandatory leakage prevention gate
6. **Validate** — confirm shape, zero missing values, correct column count
7. **Save** final dataset to Google Drive

## Engineered Features
| Feature | Formula | Purpose |
|---|---|---|
| `aggressive_score` | **[v2] Now computed in Notebook 5 after the train/test split** (scaler fitted on training clips only) | Composite driving aggression indicator |
| `env_complexity` | Weighted sum of detected object counts | Scene complexity measure |
| `object_density` | `total_objects / (env_complexity + 1)` | Normalised object density per scene |

## Critical Quality Gate
Step 5 runs a **programmatic check** confirming `anomaly_score` is absent from the dataset.
If it appears, the pipeline halts ([v2] now enforced with an exception) — this prevents Notebook 1 sampling scores leaking into Notebook 5 labels.

## Input
- `prepared_dataset_2000.csv` — from Notebook 1
- `ego_features_2000.csv` — from Notebook 2
- `visual_features_final.csv` — from Notebook 3

## Output
- `final_dataset.csv` — one row per study clip with complete ego-motion and visual features (count printed at runtime); 47 model features here, `aggressive_score` added in Notebook 5

## Feature Groups in Final Dataset
| Group | Features | Count |
|---|---|---|
| Ego / Behaviour | speed, acceleration, jerk, braking | 8 |
| Context / Time | hour, month, is_night, is_peak_hour, lighting, is_poor_lighting | 6 |
| Engineered | aggressive_score, env_complexity, object_density | 3 |
| Basic Visual | brightness, contrast | 2 |
| YOLO Object Detection | object counts per category | 9 |
| CNN Embeddings (PCA) | cnn_feature_0 to cnn_feature_9 | 10 |
| ViT Embeddings (PCA) | vit_feature_0 to vit_feature_9 | 10 |
| **Total** | | **48** |

## Next Notebook
→ **Notebook 5** loads this dataset, creates proxy risk labels, trains models, and evaluates the framework

> **Version 2** — an optimised, more robust version of the archived notebook in `notebooks/`. Changes are marked with `# [v2]` comments; see [`CHANGES.md`](../CHANGES.md).

> **How to run.** Edit only the **SETTINGS** cell below, then *Runtime → Run all*.
> On Colab, sign in to Hugging Face when asked (or add an `HF_TOKEN` secret) with an account that has
> accepted the NVIDIA dataset licence. Run the notebooks in order 1 → 5; each reads the previous one's output
> from `DATA_FOLDER`. No NVIDIA access? Try the 5-minute [demo](../demo/demo_pipeline.ipynb) instead.

In [ ]:
# ============================================================
# SETTINGS — the only cell you need to edit
# ============================================================
# DATA_FOLDER: where the pipeline_data folders (01_… to 05_…) live.
#   Colab  → a folder in your Google Drive (mounted automatically below)
#   Local  → ./pipeline_data next to the notebooks
import os, sys
IN_COLAB = "google.colab" in sys.modules
DATA_FOLDER = "/content/drive/MyDrive/driving-risk/pipeline_data" if IN_COLAB else os.path.abspath("pipeline_data")

if IN_COLAB and DATA_FOLDER.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
for sub in ("01_data_preparation", "02_ego_features", "03_visual_features", "04_dataset", "05_modelling_outputs"):
    os.makedirs(os.path.join(DATA_FOLDER, sub), exist_ok=True)
base_path = DATA_FOLDER
print("Data folder:", base_path)

In [ ]:
# ============================================================
# STEP 1: LOAD DATASETS
# ============================================================
import pandas as pd

# base_path comes from the SETTINGS cell at the top

ego_df = pd.read_csv(f"{base_path}/02_ego_features/ego_features_2000.csv")
visual_df = pd.read_csv(f"{base_path}/03_visual_features/visual_features_final.csv")
prepared_df = pd.read_csv(f"{base_path}/01_data_preparation/prepared_dataset_2000.csv")

print("Ego:", ego_df.shape)
print("Visual:", visual_df.shape)
print("Prepared:", prepared_df.shape)

In [ ]:
# ============================================================
# STEP 2: MERGE DATASETS
# ============================================================
# PURPOSE:Merge ego-motion, visual, and metadata features on clip_id.
#         anomaly_score is deliberately NOT merged in — it will be
#         created fresh in Notebook 5 on training data only.
# OUTPUT:Unified feature dataset without target leakage.

# ------------------------------------------------------------
# DROP chunk FROM EGO TO AVOID COLUMN CLASH
# ------------------------------------------------------------
ego_df_clean = ego_df.drop(columns=["chunk"], errors="ignore")

# ------------------------------------------------------------
# MERGE EGO + VISUAL
# ------------------------------------------------------------
merged_df = ego_df_clean.merge(visual_df, on="clip_id", how="inner")

print("After ego + visual merge:", merged_df.shape)

# [v2] Coverage checks — confirm every visual row belongs to the study sample.
prep_ids   = set(prepared_df["clip_id"].astype(str))
visual_ids = set(visual_df["clip_id"].astype(str))
stray = visual_ids - prep_ids
assert len(stray) == 0, (
    f"{len(stray)} visual rows are not in the study sample. Re-run Notebook 3 (v2) first."
)
print(f"Visual coverage of study sample: {len(visual_ids & prep_ids)} / {len(prep_ids)}")

# ------------------------------------------------------------
# MERGE METADATA (NO anomaly_score)
# ------------------------------------------------------------
meta_cols = [
    "clip_id", "chunk", "country", "month", "hour_of_day",
    "platform_class", "radar_config", "cluster", "scenario_type",
    "is_night", "is_peak_hour", "lighting_condition"
]

# Only keep meta_cols that actually exist in prepared_df
meta_cols_available = [c for c in meta_cols if c in prepared_df.columns]

merged_df = merged_df.merge(
    prepared_df[meta_cols_available],
    on="clip_id",
    how="inner"
)

print("After metadata merge:", merged_df.shape)
print("\nColumns:", list(merged_df.columns))
print("\nMissing values per column:")
print(merged_df.isna().sum()[merged_df.isna().sum() > 0])

print("\nFirst 5 rows:")
display(merged_df.head())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 2 complete: Datasets merged. anomaly_score excluded.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================
# PURPOSE:Create composite features from existing columns.
#         aggressive_score uses StandardScaler to normalise
#         components before summing — prevents scale dominance.
# OUTPUT:Three engineered features added to merged_df.

from sklearn.preprocessing import StandardScaler
import numpy as np

# ------------------------------------------------------------
# AGGRESSIVE SCORE — SCALED COMPONENTS
# ------------------------------------------------------------
# Normalise each component before summing to prevent
# max_speed or max_jerk dominating due to scale differences

# [v2] aggressive_score is computed in Notebook 5, Step 2, after the train/test split,
#      with its scaler fitted on training clips only.

# ------------------------------------------------------------
# ENVIRONMENTAL COMPLEXITY — WEIGHTED OBJECT COUNT
# ------------------------------------------------------------
merged_df["env_complexity"] = (
    merged_df["person_count"]        * 3.0 +
    merged_df["bicycle_count"]       * 2.0 +
    merged_df["motorcycle_count"]    * 2.0 +
    merged_df["car_count"]           * 1.0 +
    merged_df["truck_count"]         * 1.5 +
    merged_df["bus_count"]           * 1.5 +
    merged_df["traffic_light_count"] * 1.0
)

print("Feature engineering complete.")

print("\nenv_complexity stats:")
print(merged_df["env_complexity"].describe())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 3 complete: Engineered features created with proper scaling.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 4: DATA IMPROVEMENTS
# ============================================================
# PURPOSE:Add object density, clip extremes, and lighting label.
#         object_density uses total_objects per clip (not divided
#         by brightness — that formula was not defensible).
# OUTPUT:Refined dataset with additional quality features.

# ------------------------------------------------------------
# OBJECT DENSITY — TOTAL OBJECTS NORMALISED BY ENV COMPLEXITY
# ------------------------------------------------------------
# Simple, interpretable: how many objects relative to scene complexity
merged_df["object_density"] = (
    merged_df["total_objects"] /
    (merged_df["env_complexity"] + 1)  # +1 to avoid div by zero
)

# ------------------------------------------------------------
# CLIP JERK EXTREMES FOR STABILITY
# ------------------------------------------------------------
merged_df["max_jerk"] = merged_df["max_jerk"].clip(-50, 50)

# ------------------------------------------------------------
# LIGHTING CONDITION LABEL (CATEGORICAL — FOR ENCODING IN NB5)
# ------------------------------------------------------------
merged_df["lighting_condition"] = pd.cut(
    merged_df["brightness"],
    bins=[0, 40, 100, 255],
    labels=["poor", "moderate", "good"],
    include_lowest=True   # [v2] brightness exactly 0 would otherwise become NaN
)

# Binary poor lighting flag
merged_df["is_poor_lighting"] = (merged_df["brightness"] < 40).astype(int)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------
print("Data improvements applied.\n")

print("object_density stats:")
print(merged_df["object_density"].describe())

print("\nlighting_condition distribution:")
print(merged_df["lighting_condition"].value_counts())

print("\nis_poor_lighting distribution:")
print(merged_df["is_poor_lighting"].value_counts())

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 4 complete: Data improvements applied.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# STEP 5: CLEAN DATASET
# ============================================================
# PURPOSE:Remove any rows with missing values in key feature columns.
#         anomaly_score is NOT present — no leakage check needed.
# OUTPUT:Clean final dataset ready for modelling.

# ------------------------------------------------------------
# CHECK MISSING VALUES
# ------------------------------------------------------------
missing = merged_df.isna().sum()
missing = missing[missing > 0]

print("Missing values before cleaning:")
print(missing if len(missing) > 0 else "None — dataset is clean.")

# ------------------------------------------------------------
# DROP ROWS WITH ANY MISSING FEATURE VALUES
# ------------------------------------------------------------
# [v2] radar_config is metadata, not a model feature, and is often blank.
#      Fill it as "none" and drop rows only when an actual model feature is missing.
merged_df["radar_config"] = merged_df["radar_config"].fillna("none")

non_feature_cols = ["clip_id", "chunk", "country", "platform_class", "radar_config",
                    "cluster", "scenario_type"]
feature_cols = [c for c in merged_df.columns if c not in non_feature_cols]

before = len(merged_df)
merged_df = merged_df.dropna(subset=feature_cols)
after = len(merged_df)

print(f"\nRows dropped: {before - after}")
print(f"Final dataset shape: {merged_df.shape}")

# ------------------------------------------------------------
# VERIFY NO anomaly_score IN DATASET
# ------------------------------------------------------------
if "anomaly_score" in merged_df.columns:
    raise RuntimeError("anomaly_score found in dataset — remove before saving.")  # [v2] now halts
else:
    print("\n✅  Confirmed: anomaly_score NOT present. No leakage.")

# ------------------------------------------------------------
# FINAL COLUMN LIST
# ------------------------------------------------------------
print("\nFinal columns:")
print(list(merged_df.columns))

# ------------------------------------------------------------
# FINAL CONFIRMATION
# ------------------------------------------------------------
print("\n------------------------------------------------------------")
print("Step 5 complete: Dataset cleaned. No target leakage present.")
print("------------------------------------------------------------")

In [ ]:
# ============================================================
# [v2] STEP 5B: DATASET AUDIT — numbers to report in the paper
# ============================================================
print("Final clips:", len(merged_df), "of", len(prepared_df), "sampled")
print("\nScenario type (from Notebook 1 sampling):")
print(merged_df["scenario_type"].value_counts().to_string())
print("\nCountry share (%):")
print((merged_df["country"].value_counts(normalize=True) * 100).round(1).to_string())
print(f"\nShare of clips with hour_of_day = 0: {(merged_df['hour_of_day'] == 0).mean():.1%}")
print(f"Share of clips with is_night = 1:     {merged_df['is_night'].mean():.1%}")
print(f"Share of clips where max_jerk hits the ±50 clip limit: {(merged_df['max_jerk'].abs() >= 49.99).mean():.1%}")
print("  (high values mean the jerk features are dominated by finite-difference noise — report as a limitation)")


In [ ]:
# ============================================================
# STEP 6: SAVE DATASET
# ============================================================
import os

output_path = f"{base_path}/04_dataset"
os.makedirs(output_path, exist_ok=True)

csv_path = f"{output_path}/final_dataset.csv"
parquet_path = f"{output_path}/final_dataset.parquet"

merged_df.to_csv(csv_path, index=False)
merged_df.to_parquet(parquet_path, index=False)

print("Saved at:")
print(csv_path)
print(parquet_path)